In [83]:
import pandas as pd
df = pd.read_parquet("../data/cleaned/results.parquet")

In [84]:
df["speed_kmh"] = df["Distance"] / (df["temps_sec"] / 3600)
df["year"] = df['Date'].dt.year

Pour tenter de corriger les noms, on ne peut le faire que sur base du dataset.

On ne peut donc se concentrer que sur les coureurs récurrents (qu'on a vu un minimum 3 fois) et voir si dans le reste du set de données, il y a des noms très semblables pour les combiner en une seule et meme personne.

On prendra pour ca la similitude de nom, le sexe, la catégorie si la meme année, la vitesse (distance/temps).

# 1. Nettoyage des noms 
suppression des caractères spéciaux (',-,...) qui risquent de ne pas etre cohérent d'une instance à l'autre. (exemple Jean-Christophe, Jean Christophe)


In [85]:
import re

def normalize_name(name):
    name = re.sub(r"[^a-zA-Z\s]", " ", name)
    name = name.strip()
    return name

df["Nom_clean"] = df["Nom"].apply(normalize_name)

In [86]:
runner_counts = df["Nom_clean"].value_counts()
frequent_runners = runner_counts[runner_counts >= 3].index

df_freq = df[df["Nom_clean"].isin(frequent_runners)].copy()

In [87]:
len(df_freq['Nom'].unique())

7087

In [88]:
len(df_freq['Nom_clean'].unique())

6776

Cela permet deja de nettoyer 311 noms

# 2. Référentiel de nom sur les coureurs les plus fréquents

On commence par un référentiel sur base des noms de coureurs fréquents. On va d'abord regarder parmis eux si il y a des noms à fusionner.

Exemple:
- Abdullahi Mohamed 5* 
- Abdullani Mohamed 6*

Il faut donner un score de similarité basé sur:
- vitesse moyenne
- Matching du nom

Contraintes d'exclusion: sexe différent, participation à une meme course.

Si score élevé --> renommer Nom_clean avec le nom le plus présent.

In [89]:
runner_profile = (
    df_freq.groupby("Nom_clean")
    .agg(
        n_races=("Nom_clean", "size"),
        sexe=("Sexe", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        mean_speed=("speed_kmh", "mean"),
    )
    .reset_index()
)

runner_profile

,Nom_clean,n_races,sexe,mean_speed
0,A A,4,H,11.184022
1,Abascal Bruno,3,H,11.903050
2,Abdelkafi Hedi,4,H,11.677851
3,Abdelman Vincent,4,H,11.846750
4,Abdelmoumni Mohamed,4,H,12.637923
...,...,...,...,...
6771,Zielinski Michel,4,H,14.626106
6772,Zigrossi Mauro,5,H,9.440540
6773,Zimmer Dylan,4,H,13.153241
6774,Zuliani Alexandre,10,H,14.897285


In [90]:
runner_races = (
    df_freq.groupby("Nom_clean")["racekey"]
    .apply(set)
    .to_dict()
)

In [91]:
def same_race(name1, name2):

    return len(
        runner_races[name1]
        &
        runner_races[name2]
    ) > 0

In [92]:
from rapidfuzz import fuzz
from itertools import combinations

pairs = []

profiles = runner_profile.to_dict("records")

for a, b in combinations(profiles, 2):

    # sexe incompatible -> skip direct
    if a["sexe"] != b["sexe"]:
        continue

    # même course impossible
    if same_race(a["Nom_clean"], b["Nom_clean"]):
        continue

    # similarité nom
    name_score = fuzz.token_sort_ratio(a["Nom_clean"], b["Nom_clean"])

    # différence vitesse moyenne
    speed_diff = abs(a["mean_speed"] - b["mean_speed"])

    # score vitesse
    if speed_diff < 0.5:
        speed_score = 100
    elif speed_diff < 1:
        speed_score = 80
    elif speed_diff < 2:
        speed_score = 50
    else:
        speed_score = 0

    # score final pondéré
    final_score = (
        0.9 * name_score +
        0.1 * speed_score
    )

    pairs.append({
        "name_1": a["Nom_clean"],
        "name_2": b["Nom_clean"],
        "name_score": round(name_score, 1),
        "speed_diff": round(speed_diff, 2),
        "speed_score": speed_score,
        "final_score": round(final_score, 1),
        "races_1": a["n_races"],
        "races_2": b["n_races"],
    })

In [93]:
pairs_df = pd.DataFrame(pairs)

In [94]:
candidates = pairs_df[pairs_df["final_score"] >= 95]

candidates


,name_1,name_2,name_score,speed_diff,speed_score,final_score,races_1,races_2
152130,Amor Y Suarez Cesar,Amor Y Suarez Cesario,95.0,0.18,100,95.5,3,3
225564,Anne Reynders,Reynders Anne,100.0,0.46,100,100.0,7,76
914074,Benoit Lallemand,Lallemand Benoit,100.0,1.50,50,95.0,3,225
1833940,Broset Olivier,Brosset Olivier,96.6,0.05,100,96.9,8,3
2013308,Calogero Mauro,Mauro Calogero,100.0,1.17,50,95.0,4,238
2234466,Chabothier Bruno,Chabotier Bruno,96.8,0.07,100,97.1,6,3
2349802,Charlier Mathieu,Charlier Matthieu,97.0,0.43,100,97.3,9,12
2491990,Ciamara Fabien,Ciamarra Fabien,96.6,0.22,100,96.9,3,45
3081796,Corteleven Willy,Cortleven Willy,96.8,0.98,80,95.1,7,44
3530325,Darmont Theo,Theo Darmont,100.0,1.98,50,95.0,80,6


In [95]:
correction_dict = {}
for _, row in candidates.iterrows():

    n1 = row["name_1"]
    n2 = row["name_2"]

    n1_races = row["races_1"]
    n2_races = row["races_2"]

    if n1_races > n2_races:
        canonical_name = n1
        other = n2
    else:
        canonical_name = n2
        other = n1

    correction_dict[other] = canonical_name


On applique maintenant la correction sur tout le dataset (df et df_freq)

In [96]:
df_freq["Nom_clean"] = (
    df_freq["Nom_clean"]
    .replace(correction_dict)
)


In [97]:
df["Nom_clean"] = (
    df["Nom_clean"]
    .replace(correction_dict)
)

# 3. 2eme passe sur les coureurs moins fréquents

On utlise le référentiel de df_freq. 

On défini un runner profile comme précédemment, mais maintenant on recherche dans le dataframe des coureurs avec 1-2 courses

Mais vu qu'il y a un grand nombre de données, il faut réduire l'epace de comparaison sinon, on a deux boucles imbriquées qui doivent calculer 121 millions de scores de comparaison.

In [98]:
runner_counts = (df["Nom_clean"].value_counts())
frequent_runners = runner_counts[runner_counts >= 3].index

df_freq = df[df["Nom_clean"].isin(frequent_runners)].copy()

In [99]:
freq_profile = (
    df_freq.groupby("Nom_clean")
    .agg(
        n_races=("Nom_clean", "size"),
        sexe=("Sexe", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        mean_speed=("speed_kmh", "mean"),
    )
    .reset_index()
)

In [100]:
unfrequent_runners = runner_counts[runner_counts < 3].index

unfreq_df = df[df["Nom_clean"].isin(unfrequent_runners)].copy()


In [101]:
runner_races = (
    df.groupby("Nom_clean")["racekey"]
    .apply(set)
    .to_dict()
)

In [102]:
unfreq_profile = (
    unfreq_df.groupby("Nom_clean")
    .agg(
        n_races=("Nom_clean", "size"),
        sexe=("Sexe", lambda x: x.mode().iloc[0] if not x.mode().empty else None),
        mean_speed=("speed_kmh", "mean"),
    )
    .reset_index()
)

Clé de blockage pour éviter de comparer all vs all

In [103]:
# retourne la concaténation des deux premières lettres de chaque token du nom, trié (pour éviter les inversions nom/prénom)
def blocking_key(name):

    parts = sorted(name.split())

    key = "".join(
        p[:3]
        for p in parts
    )

    return key

In [104]:
freq_profile["block"] = (
    freq_profile["Nom_clean"]
    .apply(blocking_key)
)

unfreq_profile["block"] = (
    unfreq_profile["Nom_clean"]
    .apply(blocking_key)
)

In [105]:
freq_blocks = (
    freq_profile
    .groupby("block")
)

In [106]:
from collections import defaultdict

freq_index = defaultdict(list)

for _, row in freq_profile.iterrows():
    freq_index[row["block"]].append(row)

In [110]:
from rapidfuzz import process

matches = []

for _, rare in unfreq_profile.iterrows():

    block = rare["block"]

    candidates = freq_index.get(block, [])

    if not candidates:
        continue

    candidate_names = [
        c["Nom_clean"]
        for c in candidates
    ]

    results = process.extract(
        rare["Nom_clean"],
        candidate_names,
        scorer=fuzz.token_sort_ratio,
        limit=5
    )

    for matched_name, score, _ in results:

        if score < 85:
            continue

        ref = next(
            c for c in candidates
            if c["Nom_clean"] == matched_name
        )

        # hard constraints
        if rare["sexe"] != ref["sexe"]:
            continue

        # différence vitesse moyenne
        speed_diff = abs(rare["mean_speed"] - ref["mean_speed"])

        # score vitesse
        if speed_diff < 0.5:
            speed_score = 100
        elif speed_diff < 1:
            speed_score = 80
        elif speed_diff < 2:
            speed_score = 50
        else:
            speed_score = 0

        final_score = (
            0.9 * score
            + 0.1 * speed_score
        )

        matches.append({
            "rare_name": rare["Nom_clean"],
            "freq_name": matched_name,
            "final_score": round(final_score, 1),
            "races_rare": rare["n_races"],
            "races_freq": ref["n_races"],
        })


In [111]:
matches_df = pd.DataFrame(matches)

In [112]:
candidates = matches_df[matches_df["final_score"] >= 95]

candidates

,rare_name,freq_name,final_score,races_rare,races_freq
4,Albessart Francois,Albessard Francois,95.0,1,3
10,Alison Danloy,Danloy Alison,100.0,1,27
11,Aloise Giltay,Giltay Aloise,98.0,2,41
15,Andre Claud,Andre Claude,96.1,1,15
16,Andre Ehx,Ehx Andre,100.0,1,8
...,...,...,...,...,...
1776,Wilquet Anne Claude,Wiliquet Anne Claude,97.7,1,4
1783,Yasmina El Asri,El Asri Yasmina,100.0,1,3
1785,Youri De Nie,De Nie Youri,100.0,1,5
1788,Zanghellimi Claudio,Zanghellini Claudio,95.3,1,18


In [114]:
correction_dict = {}
for _, row in candidates.iterrows():

    n1 = row["rare_name"]
    n2 = row["freq_name"]

    canonical_name = n2
    other = n1

    correction_dict[other] = canonical_name


In [115]:
df["Nom_clean"] = (
    df["Nom_clean"]
    .replace(correction_dict)
)

In [116]:
runner_counts = df["Nom_clean"].value_counts()
runner_counts.describe()

count    24272.000000
mean         4.728123
std         13.090156
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        246.000000
Name: count, dtype: float64

In [117]:
df

,Position,Nom,Club,Sexe,Categorie,Position Catégorie,Temps,NomCourse,Date,Distance,source_file,racekey,categorie_cleaned,temps_clean,temps_sec,speed_kmh,year,Nom_clean
0,1,Simonet Yves,P8,H,SEN,1,0:39:48,La Neupreenne - Neupre,2017-03-12,11.0,170312-CONDRUSIEN-CHRONORACE-NEUPRE-11000.pdf,la neupreenne - neupre_2017-03-12_11.0,SH,0 days 00:39:48,2388,16.582915,2017,Simonet Yves
1,2,Honnay Xavier,,H,SEN,2,0:39:49,La Neupreenne - Neupre,2017-03-12,11.0,170312-CONDRUSIEN-CHRONORACE-NEUPRE-11000.pdf,la neupreenne - neupre_2017-03-12_11.0,SH,0 days 00:39:49,2389,16.575973,2017,Honnay Xavier
2,3,Corswarem Pierre-Henri,Foulees Du Plai,H,SEN,3,0:39:53,La Neupreenne - Neupre,2017-03-12,11.0,170312-CONDRUSIEN-CHRONORACE-NEUPRE-11000.pdf,la neupreenne - neupre_2017-03-12_11.0,SH,0 days 00:39:53,2393,16.548266,2017,Corswarem Pierre Henri
3,4,Grilo Carvalho Thomas,Foulees Du Plai,H,SEN,4,0:39:54,La Neupreenne - Neupre,2017-03-12,11.0,170312-CONDRUSIEN-CHRONORACE-NEUPRE-11000.pdf,la neupreenne - neupre_2017-03-12_11.0,SH,0 days 00:39:54,2394,16.541353,2017,Grilo Carvalho Thomas
4,5,Schmitz Bertrand,Foulees Du Plai,H,SEN,5,0:39:54,La Neupreenne - Neupre,2017-03-12,11.0,170312-CONDRUSIEN-CHRONORACE-NEUPRE-11000.pdf,la neupreenne - neupre_2017-03-12_11.0,SH,0 days 00:39:54,2394,16.541353,2017,Schmitz Bertrand
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114772,120,Vo Van Barbara,,F,DAM,22,0:49:12,Jogging De Terwagne,2015-04-17,6.3,2015-CONDRUSIEN-TERWAGNE-6.30.pdf,jogging de terwagne_2015-04-17_6.3,SD,0 days 00:49:12,2952,7.682927,2015,Vo Van Barbara
114773,121,Charlier Bernadette,,F,AI1,14,0:49:13,Jogging De Terwagne,2015-04-17,6.3,2015-CONDRUSIEN-TERWAGNE-6.30.pdf,jogging de terwagne_2015-04-17_6.3,A1,0 days 00:49:13,2953,7.680325,2015,Charlier Bernadette
114774,122,Falla Coline,,F,DAM,23,0:52:51,Jogging De Terwagne,2015-04-17,6.3,2015-CONDRUSIEN-TERWAGNE-6.30.pdf,jogging de terwagne_2015-04-17_6.3,SD,0 days 00:52:51,3171,7.152318,2015,Falla Coline
114775,123,Falla Annabelle,,F,DAM,24,0:52:53,Jogging De Terwagne,2015-04-17,6.3,2015-CONDRUSIEN-TERWAGNE-6.30.pdf,jogging de terwagne_2015-04-17_6.3,SD,0 days 00:52:53,3173,7.147810,2015,Falla Annabelle


In [119]:
len(df["Nom_clean"].unique())

24272